In [1]:
import json
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import seaborn as sns

from traccuracy.loaders import load_geff_data

merge_iter_root = '/home/ddon0001/PhD/experiments/scaled/pre-thesis/merge_resolution/'
starting_solution_root = '/home/ddon0001/PhD/experiments/scaled/pre-thesis/scaled_w_merge/'
ending_solution_root = '/home/ddon0001/PhD/experiments/scaled/pre-thesis/post_merge_resolution_merges'
all_merge_info = pd.read_csv(merge_iter_root + 'all_iterations_merge_info.csv')

ds_names_with_merges_resolved = [
    ds_name for ds_name in os.listdir(ending_solution_root) if os.path.isdir(os.path.join(ending_solution_root, ds_name))
]

In [2]:
all_merge_info.head()

,ds_name,merge_node,merge_length,merge_fate,merge_fate_correct,merge_exit_node,merge_id,parent_has_fn_succ,fn_succ,fn_overlaps,fn_overlaps_merge,iteration,merge_t,new_seg_id
0,PhC-C2DL-PSC_02,6587,0,divide,True,6587,PhC-C2DL-PSC_02_6587,False,-1,False,False,0,NaN,NaN
1,PhC-C2DL-PSC_02,10158,11,divide,False,11484,PhC-C2DL-PSC_02_10158,False,-1,False,False,0,NaN,NaN
2,PhC-C2DL-PSC_02,11607,1,divide,False,11731,PhC-C2DL-PSC_02_11607,False,-1,False,False,0,NaN,NaN
3,PhC-C2DL-PSC_02,12432,2,divide,False,12701,PhC-C2DL-PSC_02_12432,True,939_140,False,False,0,NaN,NaN
4,PhC-C2DL-PSC_02,12997,0,divide,False,12997,PhC-C2DL-PSC_02_12997,False,-1,False,False,0,NaN,NaN


In [ ]:
for ds_name in ds_names_with_merges_resolved:
    sol_path = os.path.join(
        starting_solution_root,
        ds_name,
        'matched_solution.zarr',
        'pred.geff'
    )
    starting_solution = load_geff_data(sol_path, load_all_props=True).graph
    ds_merge_ids = all_merge_info[(all_merge_info.ds_name == ds_name) & (all_merge_info.iteration == 0) & (all_merge_info.parent_has_fn_succ)]['merge_node']
    for merge_id in ds_merge_ids:
        for edge in starting_solution.in_edges(merge_id):
            edge_info = starting_solution.edges[edge]
            if 'fp' not in edge_info:
                assert 'tp' in edge_info, f"Edge {edge} has neither fp nor tp!"
            # src of fp edge SHOULD have an FN edge outgoing (but we can't know that from pred graph)
            # so we need to load the gt graph and the node mapping...
            else:
